# Лекция 4б. Итераторы и включения

## Цели лекции

После этой лекции вы сможете:
- искать и разбирать текст регулярными выражениями (`re`);
- объяснять итерационный протокол: чем итерируемый объект отличается от итератора
  и почему одни объекты выдерживают повторный проход, а другие нет;
- писать генераторы (`yield`, `yield from`) и понимать, чем ленивая выдача
  выгоднее готового списка;
- строить коллекции включениями (списковые, словарные, множественные);
- применять функциональные инструменты `functools` (кэш, `reduce`, `partial`) и `itertools`.

## Регулярные выражения: `re`

**Регулярное выражение** – шаблон для поиска и разбора текста. Модуль `re` – стандартный
инструмент: найти, извлечь части, проверить формат, заменить по шаблону.

**Шаблон читается слева направо.** Обычные символы означают сами себя,
а квадратные скобки задают **один** символ из набора.

| Запись | Что ищем |
| --- | --- |
| `AB-` | буквальный текст `AB-` |
| `[A-Z]` | одну заглавную латинскую букву |
| `[0-9]` | одну цифру от 0 до 9 |
| `[^0-9]` | один символ, кроме цифр от 0 до 9 |

`findall` покажет все найденные совпадения.


In [1]:
import re

order = "AB-2048"
print(re.findall(r"AB-", order))
print(re.findall(r"[A-Z]", order))
print(re.findall(r"[0-9]", order))
print(re.findall(r"[^0-9]", order))


['AB-']
['A', 'B']
['2', '0', '4', '8']
['A', 'B', '-']


### Собираем шаблон: сколько раз повторяется часть

Квантификатор относится к **предыдущему символу или набору**:

| Запись | Число повторений |
| --- | --- |
| `?` | ноль или один |
| `*` | ноль или больше |
| `+` | один или больше |
| `{n}` / `{m,n}` | ровно `n` / от `m` до `n` |

Номер заказа: две заглавные латинские буквы, дефис, четыре цифры.
Собираем его из частей: `[A-Z]{2}` + `-` + `[0-9]{4}`.


In [2]:
orders = "AB-2048; CD-7310; A-15"
print(re.findall(r"[A-Z]+", orders))
print(re.findall(r"[0-9]{4}", orders))
print(re.findall(r"[A-Z]{2}-[0-9]{4}", orders))


['AB', 'CD', 'A']
['2048', '7310']
['AB-2048', 'CD-7310']


### Специальные символы и экранирование

- `\d` – десятичная цифра; `[0-9]` ограничивает выбор цифрами ASCII.
- `\w` – буква, цифра или `_`; `\s` – пробельный символ.
- `.` – любой символ, кроме перевода строки.
- `\.` – обычная точка: обратный слэш снимает специальный смысл. Это называется
  **экранированием** и работает с любым спецсимволом: `\+`, `\(`, `\\`.

Поэтому шаблон имени файла с неэкранированной точкой находит лишнее.

In [3]:
files = "report.txt reportXtxt"
print(re.findall(r"report.txt", files))
print(re.findall(r"report\.txt", files))


['report.txt', 'reportXtxt']
['report.txt']


### Поиск, извлечение и замена


In [1]:
import re

text = "Телефон: +7-900-123-45-67, почта: ivan@example.com"

# search – найти первое совпадение по шаблону
m = re.search(r"\d{3}-\d{3}-\d{2}-\d{2}", text)
print("Нашли:", m.group())

# findall – все совпадения
print(re.findall(r"\d+", text))

# группы () – вытащить части совпадения
email = re.search(r"(\w+)@(\w+\.\w+)", text)
print("Логин:", email.group(1), "| Домен:", email.group(2))

# sub – замена по шаблону
print(re.sub(r"\d", "*", text))

Нашли: 900-123-45-67
['7', '900', '123', '45', '67']
Логин: ivan | Домен: example.com
Телефон: +*-***-***-**-**, почта: ivan@example.com


### Скобки меняют результат `findall`

Количество **захватывающих групп** определяет, что попадёт в список:

- нет групп – полные совпадения;
- одна группа – содержимое этой группы;
- несколько групп – кортежи с их содержимым.

Скобки могут изменить результат, даже если найдены те же участки текста.


In [1]:
import re

stock = "desk=12 shelf=7"
print(re.findall(r"\w+=\d+", stock))
print(re.findall(r"\w+=(\d+)", stock))
print(re.findall(r"(\w+)=(\d+)", stock))


['desk=12', 'shelf=7']
['12', '7']
[('desk', '12'), ('shelf', '7')]


### Группа без захвата: `(?:...)`

Иногда скобки нужны, чтобы **объединить части шаблона**, а извлекать
их отдельно не требуется. Тогда используйте `(?:...)`.

Правило, которое здесь важнее самих скобок: если в шаблоне есть захватывающие
группы, `findall` возвращает **их содержимое**, а не совпадение целиком.
`(?:...)` захватывающей группой не считается – с ней возвращается всё совпадение.

В шаблоне `\d+(?:kg|g)` знак `|` означает выбор: после числа идут либо `kg`,
либо `g`.

In [2]:
weights = "12kg 7g"

# С захватом: findall отдаёт содержимое скобок, число в результат не попадает
print("С захватом:", re.findall(r"\d+(kg|g)", weights))

# Без захвата: скобки только группируют, возвращается совпадение целиком
print("Без захвата:", re.findall(r"\d+(?:kg|g)", weights))

С захватом: ['kg', 'g']
Без захвата: ['12kg', '7g']


### Проверка формата и именованные группы

- `fullmatch` требует, чтобы шаблон покрыл **всю** строку – удобно для валидации.
- Именованные группы `(?P<имя>...)` позволяют обращаться к частям по имени, а не по номеру.

In [2]:
import re

def valid_date(s: str) -> bool:
    return re.fullmatch(r"\d{4}-\d{2}-\d{2}", s) is not None

print(valid_date("2026-08-23"))     # True
print(valid_date("23.08.2026"))     # False – формат не тот

# Именованные группы – доступ по имени
m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})", "2026-08")
print(m.group("year"), m.group("month"))

True
False
2026 08


### Сырые строки: зачем `r` перед шаблоном

В обычной строке обратный слэш служебный: Python читает `\b` как символ забоя,
`\n` – как перевод строки. Для `\d` escape-последовательности не существует,
и строка пока остаётся прежней, но полагаться на это нельзя.

Префикс `r` отключает обработку: `r"\b"` – это ровно два символа, слэш и `b`,
то есть то, что и ждёт модуль `re`.

Правило простое: шаблоны регулярных выражений всегда пишут сырыми строками.
Ошибка не даёт исключения – выражение молча ищет не то.

In [3]:
import re

# \b в обычной строке - один служебный символ, в сырой - два обычных
print(len("\b"), len(r"\b"))

text = "python питон python3"

print(re.findall(r"\bpython\b", text))   # сырая строка: граница слова
print(re.findall("\bpython\b", text))    # обычная: шаблон стал другим

1 2
['python']
[]


### Жадность квантификаторов

`*`, `+` и `?` по умолчанию **жадные**: захватывают как можно больше, а потом
отдают назад ровно столько, сколько нужно для совпадения. Поэтому `<.*>` на строке
с двумя тегами заберёт всё от первого `<` до последнего `>`.

Ленивая форма – та же запись со знаком `?` после квантификатора: `<.*?>`.
Она берёт как можно меньше.

Это первая по частоте ошибка в регулярных выражениях, и исключения она не даёт:
выражение просто находит не то.

In [4]:
html = "<b>жирный</b> и <i>курсив</i>"

print(re.findall(r"<.*>", html))    # жадно: одно совпадение на всю строку
print(re.findall(r"<.*?>", html))   # лениво: каждый тег отдельно

['<b>жирный</b> и <i>курсив</i>']
['<b>', '</b>', '<i>', '</i>']


In [5]:
line = "имя='Иван' город='Пермь'"

print(re.findall(r"'(.*)'", line))     # жадно: от первой кавычки до последней
print(re.findall(r"'(.*?)'", line))    # лениво: каждое значение отдельно

# Надёжнее запретить лишний символ, чем полагаться на ленивость
print(re.findall(r"'([^']*)'", line))

["Иван' город='Пермь"]
['Иван', 'Пермь']
['Иван', 'Пермь']


### `compile`, `finditer` и `split`

- `re.compile(шаблон)` возвращает объект с теми же методами. Функции модуля тоже
  кешируют недавно использованные шаблоны, так что дело не в скорости разбора:
  скомпилированный объект удобно назвать по смыслу и переиспользовать явно.
- `finditer` возвращает не строки, а объекты совпадений – с текстом, границами
  и группами. Это то, что нужно, когда важно, **где** нашлось.
- `re.split` режет строку по шаблону, а не по фиксированному разделителю.

In [6]:
log = "ERROR 14:02 диск; WARN 14:05 память; ERROR 14:31 сеть"

pattern = re.compile(r"(ERROR|WARN) (\d{2}:\d{2}) (\w+)")

for m in pattern.finditer(log):
    level, moment, subject = m.groups()
    print(f"{moment} {level:5} {subject:8} символы {m.start()}-{m.end()}")

14:02 ERROR диск     символы 0-16
14:05 WARN  память   символы 18-35
14:31 ERROR сеть     символы 37-53


In [7]:
# split по шаблону: разделителем может быть целый набор символов
messy = "яблоко,  груша;слива ,  вишня"
print(re.split(r"\s*[,;]\s*", messy))

# maxsplit ограничивает число разрезов
print(re.split(r"\s*[,;]\s*", messy, maxsplit=1))

['яблоко', 'груша', 'слива', 'вишня']
['яблоко', 'груша;слива ,  вишня']


### Итерации в Python
- Цикл `for` работает не только с последовательностями (списки, кортежи, строки), а с **любым итерируемым объектом** (`iterable`).  
- Итерируемые объекты бывают:  
  - **готовые коллекции** (списки, строки, кортежи) – элементы уже лежат в памяти;  
  - **ленивые источники** (генераторы, итераторы) – выдают элементы "на лету".  
- Главное условие: из объекта можно получить итератор вызовом `iter()`.  
- Итерационные инструменты Python: `for`, comprehensions, `in`, `zip`, `enumerate`, `map` и др.  

In [53]:
# Один и тот же for обходит любой итерируемый объект
for x in [10, 20, 30]:          # список
    print(x, end=" ")
print()

for x in (10, 20, 30):          # кортеж
    print(x, end=" ")
print()

for x in "text":             # строка
    print(x, end=" ")
print()

for k in dict(a=1, b=2):     # словарь: по ключам
    print(k, end=" ")
print()

10 20 30 
10 20 30 
t e x t 
a b 


### Итерационный протокол в Python
- **Итератор** – объект с методом `__next__()`, возвращающим следующий элемент.  
- Когда элементы заканчиваются → выбрасывается исключение `StopIteration`.  
- **Итерируемый объект** – тот, из которого можно получить итератор с помощью `iter()`.  
- Итератор тоже итерируем, но отдаёт сам себя: `iter(it) is it`.  
- Все циклы `for` и многие встроенные инструменты (`map`, `zip`, `enumerate`) работают через этот протокол.  

In [12]:
# Файл для примеров: три строки
with open("data.txt", "w", encoding="utf-8") as f:
    f.write("первая строка\nвторая строка\nтретья строка\n")


In [13]:
# readline читает файл по одной строке за вызов
with open("data.txt", encoding="utf-8") as f:
    print(f.readline())  # первая строка
    print(f.readline())  # вторая строка
    print(f.readline())  # третья строка
    print(f.readline())  # пустая строка (конец файла)

первая строка

вторая строка

третья строка




In [14]:
# Правильный способ – использовать for
with open("data.txt", encoding="utf-8") as f:
    for line in f:
        print(line.upper(), end="")  # читаем и печатаем по строке

ПЕРВАЯ СТРОКА
ВТОРАЯ СТРОКА
ТРЕТЬЯ СТРОКА


### Встроенные функции iter() и next()
- `next(obj)` – возвращает следующий элемент итератора (эквивалент вызова `obj.__next__()`).
- `iter(obj)` – получает итератор из итерируемого объекта; обычно для этого вызывается `obj.__iter__()`.
- Файлы в Python являются **итераторами** сами по себе → `iter(f) is f` возвращает `True`.
- Списки и другие коллекции таковыми не являются → для них обязательно нужно вызвать `iter()`.  


In [15]:
# Пример с файлом
with open("data.txt", encoding="utf-8") as f:
    print(next(f))        # первая строка
    print(next(f))        # вторая строка
    print(iter(f) is f)

первая строка

вторая строка

True


In [16]:
# Пример со списком (список не является итератором!)
L = [1, 2, 3]
try:
    next(L)  # Ошибка: список – итерируемый объект, но не итератор
except TypeError as e:
    print("Ошибка:", e)

Ошибка: 'list' object is not an iterator


In [17]:
# Нужно создать итератор через iter()
I = iter(L)
print(next(I))  # 1
print(next(I))  # 2
print(next(I))  # 3

1
2
3


In [18]:
print(next(I))

StopIteration: 

### Ручная итерация с iter() и next()
- Цикл `for` работает автоматически: сам вызывает `iter()`, затем многократно `next()`, пока не встретит `StopIteration`.
- Можно выполнить этот процесс вручную с помощью `try/except` – это эквивалент работы цикла `for`.
- У функции `next()` есть необязательный второй аргумент – **значение по умолчанию**, которое возвращается вместо исключения `StopIteration`.
- У функции `iter()` есть особый режим с **сентинелом**: `iter(callable, sentinel)` вызывает функцию до тех пор, пока результат не совпадёт с `sentinel`.


In [19]:
# Ручная итерация (эквивалент for)
I = iter(L)
while True:
    try:
        x = next(I)
    except StopIteration:
        break
    print(x ** 2, end=" ")
print()

1 4 9 


In [20]:
# Использование next() со значением по умолчанию
L = [1]
I = iter(L)
print(next(I, "end of list"))  
print(next(I, "end of list"))  

1
end of list


In [21]:
# next() + оператор := (короче запись)
I = iter([1, 2, 3])
while (x := next(I, None)) is not None:
    print(x ** 2, end=" ")
print()

1 4 9 


In [22]:
# iter() с callable и sentinel
with open("data.txt", encoding="utf-8") as f:
    I = iter(lambda: f.read(5), "")  # читать блоками по 5 символов
    for block in I:
        print(block, end="")

первая строка
вторая строка
третья строка


### Другие встроенные итерируемые объекты
- Протокол итерации поддерживают и знакомые по прошлым лекциям `range()`, `enumerate()` и `zip()`, а также словари – они отдают ключи по одному.
- Ни один из них не список: чтобы увидеть содержимое, результат материализуют через `list()` или `tuple()`.
- Итераторы можно **вкладывать** (zip поверх enumerate поверх range и т. д.).


In [54]:
# Эти инструменты возвращают не список, а объект, выдающий элементы по запросу
print(type(range(5)).__name__, type(enumerate("text")).__name__, type(zip("ab", "cd")).__name__)

# Увидеть элементы можно, только материализовав результат
print(list(range(5)))
print(list(enumerate("text")))
print(list(zip((1, 2, 3), (10, 20, 30))))

range enumerate zip
[0, 1, 2, 3, 4]
[(0, 't'), (1, 'e'), (2, 'x'), (3, 't')]
[(1, 10), (2, 20), (3, 30)]


In [26]:
# Вложенные итераторы
print(list(enumerate(range(1, 4))))
print(list(zip(enumerate(range(1, 4)), enumerate(range(5, 8)))))

[(0, 1), (1, 2), (2, 3)]
[((0, 1), (0, 5)), ((1, 2), (1, 6)), ((2, 3), (2, 7))]


In [27]:
# for с вложенными итераторами
for x in enumerate(zip(range(1, 4), range(5, 8))):
    print(x)


(0, (1, 5))
(1, (2, 6))
(2, (3, 7))


### `reversed`: где итерируемого уже недостаточно

`reversed(x)` идёт с конца, но принимает не любой итерируемый объект. Ему нужно,
чтобы объект знал свою длину и умел отдавать элемент по индексу (либо имел
собственный `__reversed__`). Генератор не умеет ни того, ни другого – отсюда `TypeError`.

Это и есть граница между **итерируемым** и **последовательностью**: первое отдаёт
элементы по одному и только вперёд, второе позволяет обратиться к любому месту.

In [28]:
# Последовательности разворачиваются: они знают длину и индексы
print(list(reversed([1, 2, 3])))
print(list(reversed("abc")))
print(list(reversed(range(5))))

def gen():
    for x in [1, 2, 3]:
        yield x

# Генератор длину не знает – развернуть его нельзя
try:
    reversed(gen())
except TypeError as e:
    print("TypeError:", e)

# Обходной путь – сначала материализовать в список
print(list(reversed(list(gen()))))

[3, 2, 1]
['c', 'b', 'a']
[4, 3, 2, 1, 0]
TypeError: 'generator' object is not reversible
[3, 2, 1]


### Функциональные итераторы: map и filter
- **`map(func, iterable)`** – применяет функцию к каждому элементу, возвращает итератор.
- **`filter(func, iterable)`** – оставляет элементы, для которых функция вернула истину.
- Оба ленивы: пока результат не начали обходить, ни одного вызова `func` не произошло.
- Оба возвращают однопроходный итератор – про это следующий раздел.

In [29]:
# map: применяем ord() ко всем символам
M = map(ord, "py3")
print(list(M))  

[112, 121, 51]


In [30]:
# filter: оставляем только "правдивые" элементы
data = ["lp6e", "", "2024"]
print(list(filter(bool, data)))        
print(list(filter(str.isdigit, data))) 

['lp6e', '2024']
['2024']


### Однопроходные и многопроходные источники
- **Многопроходные** (`range`, `list`, `dict`): `iter()` каждый раз даёт новый итератор, поэтому независимых проходов может быть сколько угодно.  
- **Однопроходные** (`enumerate`, `zip`, `map`, `filter`, `open`): сами себе итераторы, второго прохода не будет – объект придётся создать заново.  


In [31]:
# Многопроходный источник: каждый iter(R) даёт новый итератор
R = range(3)
I1, I2 = iter(R), iter(R)
print(next(I1))  # первый проход
print(next(I2))  # второй проход начинается с начала
print(next(I1))  # первый проход продолжается

0
0
1


In [32]:
# Однопроходные итераторы
Z = zip((1, 2, 3), (10, 11, 12))
I1, I2 = iter(Z), iter(Z)      # zip сам себе итератор: I1 и I2 – один объект
print(next(I1))
print(next(I1))
print(next(I2))  # продолжаем тот же проход, а не начинаем новый

(1, 10)
(2, 11)
(3, 12)


### Проверка `in` продвигает итератор

Чтобы найти значение, `in` читает итератор до совпадения.
Прочитанные элементы, **включая найденный**, уже израсходованы.

Если значение не найдено, конечный итератор будет исчерпан.
Проверка наличия перед основным обходом может оставить ему только часть данных.


In [3]:
search_values = iter([1, 3, 4, 5, 6])
print("Нашли 4:", 4 in search_values)
print("Осталось:", list(search_values))

missing_values = iter([1, 3, 5])
print("Нашли 4:", 4 in missing_values)
print("Осталось:", list(missing_values))


Нашли 4: True
Осталось: [5, 6]
Нашли 4: False
Осталось: []


### Генераторы подробнее: ленивость и экономия памяти
- Генератор (`yield`) и его одноразовость разбирались в лекции 4. Здесь – чем ленивая выдача выгодна по памяти.  
- **Ленивость**: элемент вычисляется в момент запроса, поэтому генератор не хранит всю последовательность целиком.  
- Один и тот же расчёт, материализованный в список и оставленный ленивым, различается по памяти в десятки тысяч раз.  


In [57]:
import sys


def squares(n):
    for x in range(n):
        yield x * x


# Тот же расчёт: готовый список против ленивого генератора
all_squares = list(squares(1_000_000))
lazy_squares = squares(1_000_000)

print("Тип списка:    ", type(all_squares).__name__)
print("Тип генератора:", type(lazy_squares).__name__)
print("Память под список:   ", sys.getsizeof(all_squares), "байт")
print("Память под генератор:", sys.getsizeof(lazy_squares), "байт")

# Генератор можно передать в любую функцию, ожидающую итерируемое
print("Сумма первых десяти:", sum(squares(10)))

Тип списка:     list
Тип генератора: generator
Память под список:    8448728 байт
Память под генератор: 200 байт
Сумма первых десяти: 285


### `if generator` не проверяет наличие элементов

Генератор считается истинным даже после завершения. Вызов `bool(g)`
не получает следующий элемент и не проверяет, есть ли он.

Поэтому `if g:` не подходит для проверки пустоты генератора.
Получить элемент или значение по умолчанию позволяет уже знакомый `next`,
но получение элемента расходует его.


In [4]:
def one_value():
    yield 4


remaining_values = one_value()
print("До чтения:", bool(remaining_values))
print("Получили:", next(remaining_values))
print(next(remaining_values, "Элементов нет"))
print("После завершения:", bool(remaining_values))

До чтения: True
Получили: 4
Элементов нет
После завершения: True


### Ошибка возникает при запросе элемента

В генераторе `to_numbers` преобразование `int(text)` стоит внутри цикла,
поэтому выполняется при получении очередного элемента, а не при создании
генератора.

Поэтому `try/except` нужен **вокруг чтения**, а не только создания генератора.
Исключение, вышедшее из генератора, завершает его: внешний обработчик
не позволяет продолжить со следующей записи.


In [4]:
def to_numbers(source):
    for text in source:
        yield int(text)


numbers = to_numbers(["10", "не число", "30"])
print("Генератор создан")
print(next(numbers))
try:
    print(next(numbers))
except ValueError:
    print("Ошибка преобразования при next()")
print(next(numbers, "Генератор завершён"))

Генератор создан
10
Ошибка преобразования при next()
Генератор завершён


### Пропускаем неверную запись внутри генератора

Если по условию задачи плохую запись нужно пропустить, обрабатываем
её ошибку **внутри цикла генератора**. Тогда обработка следующей записи
продолжится.

Перехватываем только ожидаемый `ValueError` от преобразования строки.
Остальные ошибки не скрываем.


In [5]:
def parsed_integers(records):
    for record in records:
        try:
            value = int(record)
        except ValueError:
            print(f"Пропущена запись: {record!r}")
            continue
        yield value


print(list(parsed_integers(["12", "нет данных", "35"])))


Пропущена запись: 'нет данных'
[12, 35]


### Бесконечные генераторы и сохранение состояния
- Поскольку значения производятся по требованию, генератор может описывать **бесконечную** последовательность – мы берём из неё ровно столько, сколько нужно.
- Между `yield` генератор «замораживает» все свои локальные переменные – на этом строится, например, генерация чисел Фибоначчи без хранения всей истории.

In [34]:
from itertools import islice


# Бесконечный генератор: значения никогда не кончаются.
# Готовый аналог – itertools.count, он появится в разделе про itertools
def count_from(start):
    n = start
    while True:
        yield n
        n += 1


gen = count_from(10)
print(next(gen), next(gen), next(gen))          # 10 11 12

# islice берёт только первые k значений из бесконечного потока
print("Первые 5:", list(islice(count_from(100), 5)))

10 11 12
Первые 5: [100, 101, 102, 103, 104]


In [35]:
# Генератор "запоминает" состояние (a, b) между вызовами next()
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b   # обе формулы считаются на старых значениях


fib = fibonacci()
print([next(fib) for _ in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### `yield from`: передать выдачу другому итератору

Генератор часто должен отдать наружу всё, что выдаёт другой итерируемый объект.
Писать для этого цикл с `yield` необязательно: `yield from источник` делает ровно
это – перебирает источник и отдаёт его элементы по одному.

Запись короче, а на вложенных структурах избавляет от лишнего цикла внутри рекурсии.

In [36]:
def chain_manual(*sources):
    for source in sources:
        for item in source:      # ручной перебор каждого источника
            yield item


def chain_short(*sources):
    for source in sources:
        yield from source        # то же самое одной строкой


print(list(chain_manual([1, 2], "ab")))
print(list(chain_short([1, 2], "ab")))

[1, 2, 'a', 'b']
[1, 2, 'a', 'b']


In [37]:
# Рекурсивный обход вложенных списков
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # отдаём всё, что выдаст вложенный вызов
        else:
            yield item


print(list(flatten([1, [2, [3, 4]], 5])))

[1, 2, 3, 4, 5]


## Comprehensions (включения)

**Включение** (англ. *comprehension*) – компактный способ построить список,
словарь или множество из другого итерируемого. Идиоматичная для Python замена
циклу с `append` и функциям `map`/`filter`.

In [55]:
nums = [1, 2, 3, 4, 5, 6]

# Списковое включение вместо цикла с append
squares = [n * n for n in nums]
print(squares)

# С условием (аналог filter): только чётные
evens = [n for n in nums if n % 2 == 0]
print(evens)

# Отбор и преобразование в одном включении
even_squares = [n * n for n in nums if n % 2 == 0]
print(even_squares)

# Ровно то же самое через map/filter – многословнее и менее читаемо
via_mf = list(map(lambda n: n * n, filter(lambda n: n % 2 == 0, nums)))
print(via_mf)

[1, 4, 9, 16, 25, 36]
[2, 4, 6]
[4, 16, 36]
[4, 16, 36]


### Два места для `if`: отбор или выбор результата

`[x for x in data if условие]` **отбрасывает** неподходящие элементы.

`[a if условие else b for x in data]` **выбирает значение** для каждого
элемента: длина результата совпадает с длиной входа.

Во втором случае `if ... else ...` – часть выражения **перед** `for`.


In [6]:
temperatures = [-5, 0, 8, -2]
positive = [t for t in temperatures if t > 0]
nonnegative = [t if t > 0 else 0 for t in temperatures]
print("Оставили положительные:", positive)
print("Заменили остальные нулями:", nonnegative)


Оставили положительные: [8]
Заменили остальные нулями: [0, 0, 8, 0]


### Фильтр проверяется до вычисления результата

В записи `[выражение for x in data if условие]` сначала берётся `x`,
затем проверяется условие и только после этого вычисляется выражение.

Так можно отсеять значения, на которых вычисление завершилось бы ошибкой.


In [7]:
divisors = [2, 0, -4, 5]
reciprocals = [1 / x for x in divisors if x != 0]
print(reciprocals)


[0.5, -0.25, 0.2]


### Словарные, множественные и вложенные включения

Тот же синтаксис строит `dict` (`{k: v for ...}`) и допускает вложенность.
Множественные включения (`{x for ...}`) разбирались в лекции про числовые типы
вместе с множествами – здесь они просто используются.

Круглые скобки вместо квадратных дают уже знакомое генераторное выражение –
ленивый аналог спискового включения.

In [39]:
words = ["hello", "world", "python"]

# Словарное включение: слово -> его длина
print({w: len(w) for w in words})

# Множественное включение: уникальные первые буквы (sorted – для стабильного вывода)
print(sorted({w[0] for w in words}))

# Вложенное включение: матрица 3x3
matrix = [[row * 3 + col for col in range(3)] for row in range(3)]
print(matrix)

# Разворачивание вложенного списка в плоский
print([x for row in matrix for x in row])

{'hello': 5, 'world': 5, 'python': 6}
['h', 'p', 'w']
[[0, 1, 2], [3, 4, 5], [6, 7, 8]]
[0, 1, 2, 3, 4, 5, 6, 7, 8]


### Несколько `for` читаются слева направо

Первый `for` – внешний цикл, второй – вложенный. Внутренний источник
может зависеть от переменной внешнего цикла.

Ниже нужны пары индексов, у которых `column < row`: элементы под
главной диагональю таблицы. Порядок пар совпадает с порядком обхода циклами.


In [8]:
below_diagonal = [(row, column) for row in range(3) for column in range(row)]

# Те же пары через вложенные циклы
via_loops = []
for row in range(3):
    for column in range(row):
        via_loops.append((row, column))

print(below_diagonal)
print("Порядок совпал:", below_diagonal == via_loops)


[(1, 0), (2, 0), (2, 1)]
Порядок совпал: True


## `any` и `all`: проверка по всему потоку

- `any(iterable)` – истина, если истинен хотя бы один элемент; `all(iterable)` – если все.
- Обе останавливаются на решающем элементе: `any` – на первом истинном, `all` – на первом
  ложном. Дальше источник не читается.
- На бесконечном источнике это выручает, только если решающий элемент встретится:
  `any(repeat(False))` и `all(repeat(True))` не завершатся никогда.
- Если генераторное выражение – единственный аргумент, лишние скобки не нужны:
  `any(x > 0 for x in nums)`.
- На пустом входе `any([])` даёт `False`, а `all([])` – `True`. Второе не каприз:
  утверждение «ни один элемент не нарушает условие» для пустого входа верно.

In [40]:
nums = [1, 3, 5, 8, 9]

print(any(n % 2 == 0 for n in nums))   # есть ли хотя бы одно чётное
print(all(n > 0 for n in nums))        # все ли положительные

print(any([]), all([]))                # False True

True
True
False True


In [41]:
# Короткое замыкание видно по печати: проверяются не все элементы
def check(n):
    print("  проверяем", n)
    return n % 2 == 0

print("any:", any(check(n) for n in [1, 3, 4, 5, 6]))
print("all:", all(check(n) for n in [2, 4, 5, 6]))

  проверяем 1
  проверяем 3
  проверяем 4
any: True
  проверяем 2
  проверяем 4
  проверяем 5
all: False


## Функциональные инструменты: `functools` и `itertools`

Два модуля стандартной библиотеки для «функционального» стиля. `functools` работает с самими
функциями (кэш, свёртка, фиксация аргументов), `itertools` – с итераторами (ленивые комбинации
последовательностей).

Обзор `functools` ниже идёт на опережение: он держится на декораторах и передаче функций
как значений, а подробно это разбирается в лекциях 5 и 6. Сейчас достаточно знать,
что такие инструменты есть.

In [49]:
from functools import lru_cache, reduce, partial

@lru_cache                     # кэширует результаты – быстрые повторные вызовы
def fib(n): return n if n < 2 else fib(n - 1) + fib(n - 2)
print("fib(30):", fib(30))

print("reduce (произведение):", reduce(lambda a, b: a * b, [1, 2, 3, 4, 5]))

from_binary = partial(int, base=2)   # зафиксировать именованный аргумент base
print("partial from_binary('1010'):", from_binary("1010"))

fib(30): 832040
reduce (произведение): 120
partial from_binary('1010'): 10


### `itertools`: ленивые инструменты

«Ленивый» здесь не значит «почти не занимает памяти»: `cycle` запоминает
пройденные элементы, чтобы повторить их, а `product` читает входные
последовательности целиком ещё до первого результата.

Четыре группы, которые нужны чаще всего:
- бесконечные источники: `count`, `cycle`, `repeat`;
- склейка и соседние пары: `chain`, `zip_longest`, `pairwise`;
- группировка подряд идущих элементов: `groupby`;
- комбинаторика: `product`, `permutations`, `combinations`.

In [43]:
from itertools import count, cycle, repeat, islice

# count – счётчик без конца: задаются начало и шаг
print(list(islice(count(10, 5), 4)))

# cycle – бесконечно повторяет саму последовательность
print(list(islice(cycle("ABC"), 7)))

# repeat – одно и то же значение, при желании заданное число раз
print(list(repeat("x", 3)))

[10, 15, 20, 25]
['A', 'B', 'C', 'A', 'B', 'C', 'A']
['x', 'x', 'x']


In [44]:
from itertools import chain, zip_longest, pairwise

# chain – проход по нескольким последовательностям как по одной
print(list(chain([1, 2], [3, 4], [5])))

# zip_longest идёт до конца самой длинной и подставляет заполнитель
print(list(zip_longest("abc", [1, 2], fillvalue="-")))

# pairwise – соседние пары: сразу считаются разности и переходы
temps = [12, 15, 14, 19]
print([b - a for a, b in pairwise(temps)])

[1, 2, 3, 4, 5]
[('a', 1), ('b', 2), ('c', '-')]
[3, -1, 5]


### `groupby`: группы из соседних элементов

`groupby` не собирает группы по всему входу. Он идёт по элементам подряд и начинает
новую группу каждый раз, когда значение ключа изменилось. Одинаковые ключи,
разбросанные по входу, дадут несколько групп с одним и тем же ключом – и ошибки
при этом не будет.

Отсюда правило: если нужна одна группа на каждое значение ключа, вход сортируют
тем же ключом, по которому группируют. А если нужны серии подряд идущих одинаковых
элементов, сортировать нельзя: она уничтожит тот самый порядок, который мы ищем.

Вторая деталь: группа – это итератор, который живёт только до перехода к следующей.
Если группы нужны дальше, их превращают в списки сразу.

In [45]:
from itertools import groupby

words = ["арбуз", "банан", "апельсин", "вишня", "абрикос"]
def first_letter(word):
    return word[0]

# Без сортировки ключ "а" встречается трижды: между словами на "а" попали другие
for key, grp in groupby(words, key=first_letter):
    print(key, list(grp))

print()

# После сортировки тем же ключом группы получаются полными
for key, grp in groupby(sorted(words, key=first_letter), key=first_letter):
    print(key, list(grp))

print()

# А для серий подряд идущих элементов сортировка, наоборот, испортила бы данные
print([(key, len(list(grp))) for key, grp in groupby("AABBA")])

а ['арбуз']
б ['банан']
а ['апельсин']
в ['вишня']
а ['абрикос']

а ['арбуз', 'апельсин', 'абрикос']
б ['банан']
в ['вишня']

[('A', 2), ('B', 2), ('A', 1)]


In [46]:
from itertools import product, permutations, combinations

sizes = ["S", "M"]
colors = ["красный", "синий"]

# product – все сочетания по одному элементу из каждой последовательности
print(list(product(sizes, colors)))

# permutations – размещения, порядок важен: ("a", "b") и ("b", "a") различаются
print(list(permutations("abc", 2)))

# combinations – сочетания, порядок не важен
print(list(combinations("abc", 2)))

[('S', 'красный'), ('S', 'синий'), ('M', 'красный'), ('M', 'синий')]
[('a', 'b'), ('a', 'c'), ('b', 'a'), ('b', 'c'), ('c', 'a'), ('c', 'b')]
[('a', 'b'), ('a', 'c'), ('b', 'c')]


In [47]:
from itertools import batched

# batched режет поток на куски заданного размера (добавлен в Python 3.12)
for chunk in batched(range(10), 4):
    print(chunk)          # последний кусок короче, если элементы кончились

(0, 1, 2, 3)
(4, 5, 6, 7)
(8, 9)


## Обработка строк в несколько ленивых этапов

Источник → убрать пробелы → отсеять пустые строки и комментарии →
преобразовать в числа. Каждый этап получает элементы от предыдущего
**по запросу**; промежуточные списки не нужны.

Вход содержит целые числа, пустые строки и комментарии с `#`.
Печать в источнике покажет, когда очередная строка действительно прочитана.


In [9]:
def source_lines():
    for line in [" 10 ", "", "# измерения", " 20 ", "30", "40"]:
        print(f"Читаем: {line!r}")
        yield line


stripped_lines = (line.strip() for line in source_lines())
data_lines = (line for line in stripped_lines if line and not line.startswith("#"))
measurements = (int(line) for line in data_lines)
print("Этапы собраны")


Этапы собраны


### Потребитель определяет объём работы

Берём три результата через уже знакомый `islice`. Для этого потребуется
прочитать пять исходных строк: две из них отсеются.

Строка `"40"` пока не понадобится. Один запрос к последнему этапу может
вызвать несколько запросов к источнику, если фильтр пропускает не всё.


In [10]:
from itertools import islice

first_three = list(islice(measurements, 3))
print("Первые три значения:", first_three)

print("Запрашиваем ещё одно:")
print(next(measurements))


Читаем: ' 10 '
Читаем: ''
Читаем: '# измерения'
Читаем: ' 20 '
Читаем: '30'
Первые три значения: [10, 20, 30]
Запрашиваем ещё одно:
Читаем: '40'
40


## Частые ошибки

**Удаление или вставка элементов списка во время итерации по нему.** Цикл идёт по
индексам и после `remove` пропускает элементы; исключения при этом не будет. Замена
элемента по индексу безопасна. Итерируйте по копии или стройте новый список.

**Изменение состава словаря во время итерации.** Добавление или удаление ключей меняет
размер словаря, и Python выбрасывает `RuntimeError`. Замена значения у существующего
ключа безопасна. Собирайте ключи на удаление в отдельный список и удаляйте после цикла.

**Включение ради побочного эффекта.** `[print(x) for x in items]` строит и тут же
выбрасывает список из `None`. Если результат не нужен, нужен обычный цикл.

In [48]:
nums = [2, 4, 6, 8]
for n in nums:
    if n % 2 == 0:
        nums.remove(n)       # мутируем во время итерации!
print("баг:", nums)          # [4, 8] – пропустил элементы

# Правильно: построить новый список включением
nums = [2, 4, 6, 8]
nums = [n for n in nums if n % 2 != 0]
print("правильно:", nums)    # []

баг: [4, 8]
правильно: []


In [49]:
scores = {"Иван": 5, "Анна": 3, "Пётр": 4}

# Удаление прямо во время обхода словаря
try:
    for name, score in scores.items():
        if score < 4:
            del scores[name]
except RuntimeError as e:
    print("RuntimeError:", e)

# Правильно: сначала собрать ключи, потом удалить
scores = {"Иван": 5, "Анна": 3, "Пётр": 4}
to_drop = [name for name, score in scores.items() if score < 4]
for name in to_drop:
    del scores[name]
print("правильно:", scores)

RuntimeError: dictionary changed size during iteration
правильно: {'Иван': 5, 'Пётр': 4}


## Мини-упражнения


### 1. Цены со скидкой

Дан список `prices = [1200, 450, 3000, 780, 5000]`.
Одним включением постройте список цен со скидкой 10 %, округлённых до целого,
и только для товаров дороже 1000.

Ожидаемый результат: `[1080, 2700, 4500]`.

In [56]:
prices = [1200, 450, 3000, 780, 5000]

discounted = [round(price * 0.9) for price in prices if price > 1000]
print(discounted)

[1080, 2700, 4500]


### 2. `countdown(n)`

Генератор, выдающий числа от `n` до 1 (через `yield`).

In [51]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

print(list(countdown(5)))

[5, 4, 3, 2, 1]


### 3. `extract_numbers(text)`

Напишите функцию, которая находит в строке все числа (группы цифр, идущих
подряд) и возвращает их списком в порядке появления. Каждое число приведите
к `int`: нужен список целых чисел, а не строк.

Ожидаемый результат: `extract_numbers("в 2026 году 12 лекций и 5 докладов")` → `[2026, 12, 5]`.

In [52]:
# Решение: находим группы цифр и приводим каждую к int
import re


def extract_numbers(text):
    return [int(n) for n in re.findall(r"\d+", text)]


print(extract_numbers("в 2026 году 12 лекций и 5 докладов"))

[2026, 12, 5]


### 4. Первые два длинных названия

Даны строки `["  дуб ", "", " берёза ", "  ", "сосна", "ель"]`.
Удалите пробелы по краям и выберите первые два названия длиной не меньше пяти.

Постройте обработку генераторными выражениями без промежуточных списков.
Результат можно собрать в список: `["берёза", "сосна"]`.

Как изменится ответ, если ограничить вход двумя строками **до** фильтрации?


In [11]:
# Решение: ограничиваем число подходящих названий
from itertools import islice

raw_names = ["  дуб ", "", " берёза ", "  ", "сосна", "ель"]
clean_names = (name.strip() for name in raw_names)
long_names = (name for name in clean_names if len(name) >= 5)
print(list(islice(long_names, 2)))

# Если сначала ограничить вход, подходящие названия до фильтра не дойдут
early_names = (name.strip() for name in islice(raw_names, 2))
print([name for name in early_names if len(name) >= 5])


['берёза', 'сосна']
[]


## Что почитать

- [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) – лучшее введение
  в регулярные выражения; [`re`](https://docs.python.org/3/library/re.html) – справочник.
- [`functools`](https://docs.python.org/3/library/functools.html) и
  [`itertools`](https://docs.python.org/3/library/itertools.html) – рабочие инструменты
  функционального стиля.
